# Accuracy Measure Pipeline

In [40]:
import pandas as pd
import re
import numpy as np
from thefuzz import fuzz

pd.set_option('display.max_colwidth', None)

In [12]:
path_to_gt_161_164 = '../data/ground_truth/gt_161-164.csv'
path_to_gt_165_167 = '../data/ground_truth/gt_165-167.csv'
path_to_int_arch_text = '../data/A-O raw lines.csv'
path_to_r1_extract161 = '../data/a_sample/extraction/ocr_lines.txt'

## Ground Truth Table Merge

In [5]:
gt_161_164 = pd.read_csv(path_to_gt_161_164)
gt_165_167 = pd.read_csv(path_to_gt_165_167)

In [51]:
gt_df = pd.concat([gt_161_164, gt_165_167], ignore_index=True)

In [68]:
gt_df

,Year,Page,Surname,Given name,Occupation (as printed),Employer / affiliation,Work address (raw),Home address (raw),Residence marker,Transcription (working),Review status,Review notes
0,1916,161,NaN,NaN,insulated wire,A A Wire Co,93 Fed-eral,NaN,NaN,A A Wire Co insulated wire and cables 93 Fed-eral,needs review,advertisement
1,1916,161,NaN,NaN,children's dresses,A B Mfg Co,27 Kingston,NaN,NaN,"A B Mfg Co (Abe Borans, H F Brennan) children's dresses 27 Kingston",needs review,NaN
2,1916,161,NaN,NaN,warehouse,NaN,NaN,NaN,NaN,A B SEE ELECTRIC ELEVATOR CO J R Jackson jr mgr 4th floor 111 Devonshire warehouse 165 Oliver—See page 2822,needs review,bold
3,1916,161,NaN,NaN,music,A D H Publishing Co,40 State rm 41,NaN,NaN,A D H Publishing Co music 40 State rm 41,needs review,NaN
4,1916,161,NaN,NaN,paper warehouse,A STORRS & BEMENT CO,140-150 Federal,NaN,NaN,A STORRS & BEMENT CO paper warehouse 140- 150 Federal—See page 3033,needs review,bold; extra marker
...,...,...,...,...,...,...,...,...,...,...,...,...
899,1916,167,Seaborn Dor,John E,NaN,NaN,24 Hunneman Rox,at Melrose Hids {Rox,h,« John E 3 carriage mfr 24 Hunneman Rox h at Melrose Hids {Rox,NaN,NaN
900,1916,167,Seaborn Dor,John F,barber,NaN,4228 Mass ay,31 Bainbridge,h,« John F barber 4228 Mass ay h 31 Bainbridge,NaN,NaN
901,1916,167,Seaborn Dor,John F,NaN,NaN,NaN,310 Paris E B,h,« John F pedler h 310 Paris E B,NaN,NaN
902,1916,167,Seaborn Dor,John H,clerk,NaN,NaN,11 Foss Chsn {SB,h,« John H clerk h 11 Foss Chsn {SB,NaN,NaN


## Clean up plain text results from Internet Archive

In [38]:
ia_df = pd.read_csv(path_to_int_arch_text)
ia_df = ia_df.drop(columns='Unnamed: 0')
ia_df.head()

,raw_line
0,A A Wire Co insulated wire and cables 93 Fed-
1,"A B Mfg Co (Abe Borans, H F Brennan) chil-"
2,dren's dresses 27 Kingston
3,A B SEE ELECTRIC ELEVATOR CO J R
4,Jackson jr mgr 4th floor 111 Devonshire


In [92]:
import pandas as pd

# Check if a line starts with 'A' OR starts with a non-alphanumeric character (e.g. ", «, #, etc.)
# ^[^a-zA-Z0-9] matches any string that begins with a character that is NOT a letter or digit
starts_with_a = ia_df['raw_line'].str.startswith('A')
starts_with_strange = ia_df['raw_line'].str.contains(r'^[^a-zA-Z0-9]', regex=True, na=False)

# Increment group ID on 'A' lines OR strange character lines
group_start = starts_with_a | starts_with_strange
group_ids = group_start.cumsum()

# Group and combine lines
ia_df_cleaned_lines = (
    ia_df.groupby(group_ids, as_index=False)
    .agg({'raw_line': ' '.join})
    .reset_index(drop=True)
)

ia_df_cleaned_lines.head()

,raw_line
0,A A Wire Co insulated wire and cables 93 Fed-
1,"A B Mfg Co (Abe Borans, H F Brennan) chil- dren's dresses 27 Kingston"
2,A B SEE ELECTRIC ELEVATOR CO J R Jackson jr mgr 4th floor 111 Devonshire warehouse 163 Oliver — See page 2822
3,A D H Publishing Co music 40 State rm 41
4,A STORRS & BEMENT CO paper warehouse 140-150 Federal— See page 3033


In [97]:
ia_df_cleaned_lines.shape

(143034, 1)

In [98]:
gt_df.shape

(904, 12)

In [99]:
ia_df_subset = ia_df_cleaned_lines.iloc[:904].copy()

# Compare 'raw_line' from df1 against 'raw_line' from df2 row-by-row
scores = [
    fuzz.token_sort_ratio(str(str1), str(str2)) 
    for str1, str2 in zip(ia_df_subset['raw_line'], gt_df['Transcription (working)'])
]

# Create a comparison DataFrame
comparison_df = pd.DataFrame({
    'ia_line': ia_df_subset['raw_line'],
    'gt_line': gt_df['Transcription (working)'],
    'similarity_score': scores
})

# Filter to see only lines that differ significantly (e.g., less than 90% match)
mismatches = comparison_df[comparison_df['similarity_score'] < 90]

In [100]:
mismatches.head()

,ia_line,gt_line,similarity_score
6,Aab Co real estate 19 Milk rm 41 [55 Kilby,Aab Co real estate 19 Milk rm 41,88
8,Aalto J tailor h 61 Myrtle [ren Rox,Aalto J tailor h 61 Myrtle,87
15,• Herman mgr 628 E Fourth S B bds 737 E,""" Herman mgr 628 E Fourth S B bds 737 E [Broadway do",86
21,• Sophie M h 578 Newbury,""" Wolf W mgr 1022 Wash h 3 Normandy Rox",37
22,""" S E Co patent dealers 60 State rm 316",""" & Bartlett (David J Aaron, Charles E Bart-lett) lawyers 40 Court rm 1002",40


In [101]:
mismatches['similarity_score'].mean()

np.float64(35.87697516930022)

## Clean up OCR Results

In [20]:
import pandas as pd
import re

parsed_records = []
pattern = re.compile(r"^([LR]\d+)\s+\[(.*?)\]\s+(.*)$")

# Open and read the file line by line
with open(path_to_r1_extract161, "r", encoding="utf-8") as file:
    for line in file:
        match = pattern.match(line.strip())
        if match:
            line_id = match.group(1)
            coords = match.group(2)
            text_content = match.group(3).strip()
                
            parsed_records.append([line_id, coords, text_content])

# Convert into a pandas DataFrame
r1_ocr_161 = pd.DataFrame(parsed_records, columns=["Line ID", "Coordinates", "Text"])


In [21]:
r1_ocr_161.head()

,Line ID,Coordinates,Text
0,L001,"78,1662,650,20",A A Wire Co insulated wire and cables 93 Fed-
1,L002,"78,1708,651,23","A B Mfg Co (Abe Borans, H F Brennan) chil-"
2,L003,"243,1731,259,22",dren's dresses 27 Kingston
3,L004,"81,1754,647,20",A B SEE ELECTRIC ELEVATOR CO J R
4,L005,"153,1777,577,22",Jackson jr mgr 4th floor 111 Devonshire
